# Last.fm batch recommendations

Batch run Last.fm album recommender for a list of albums: runs `recommend_album()` over a subset of `albums.csv` and write results to `datasets/lastfm_recommendations_<subset>_<strategy>.csv`.

Each recommendation is one row with a `rank` column, plus Last.fm top tags and listener counts for seed and recommended albums. Outputs also include top tags (`seed_tags`, `rec_tags`).

Re-run the batch cell to resume — albums with `status` already set are skipped. Older CSVs without tag columns are still supported; missing columns are filled with `NA` on load.

**Backfill section (below):** priority backfill for `lastfm_recommendations_all_top_listener.csv` (seeds + recs), then remaining `albums.csv` gaps — checkpointing every `SAVE_EVERY` albums. Re-run to resume.

In [1]:
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from lastfm_albums import (
    format_album_tags,
    get_album_listeners_and_tags,
    recommend_album,
    set_request_delay,
)

In [2]:
DATA_DIR = Path("../datasets")
SOURCE_PATH = DATA_DIR / "albums.csv"

TRACK_SELECTION = "top_listener"  # top_listener | random | top_n_tracks
TOP_N = 3           # number of seed tracks used by top_n_tracks strategy
N_RECS = 5          # recommendations to return per album
FETCH_FLOOR = 20    # similar tracks fetched per seed track (larger = more candidates, more API calls)
REQUEST_DELAY_SEC = .7   # seconds between API requests
RANDOM_SEED = 45    # seed for random track selection (strategy "random")
SAVE_EVERY = 10     # checkpoint to CSV every N albums

## Subset

Pick which albums to process. Output file name follows the subset key and `TRACK_SELECTION`.

In [3]:
SUBSETS = {
    "all": lambda df: df,
    "3plus": lambda df: df.loc[df["review_count"] > 2],
    "test": lambda df: df.sample(250),
    "2k": lambda df: df.sample(2000),
}

In [4]:
SUBSET = "all" 
MAX_ALBUMS = None  # e.g. None for all, 10 for a dry run

In [5]:
albums = SUBSETS[SUBSET](pd.read_csv(SOURCE_PATH))
if MAX_ALBUMS is not None:
    albums = albums.head(MAX_ALBUMS)

strategy_key = (
    f"top_n_tracks_{TOP_N}" if TRACK_SELECTION == "top_n_tracks" else TRACK_SELECTION
)
OUTPUT_PATH = DATA_DIR / f"lastfm_recommendations_{SUBSET}_{strategy_key}.csv"
print(f"Subset: {SUBSET} ({len(albums):,} albums)")
print(f"Strategy: {TRACK_SELECTION}")
print(f"Output: {OUTPUT_PATH}")

Subset: all (30,516 albums)
Strategy: top_listener
Output: ../datasets/lastfm_recommendations_all_top_listener.csv


In [6]:
OUTPUT_COLS = [
    "album_id", "artist", "album", "review_count",
    "strategy", "status", "error", "seed_track", "seed_listeners", "seed_tags",
    "rec_artist", "rec_album", "score", "rank", "rec_listeners", "rec_tags",
]

results = pd.read_csv(OUTPUT_PATH) if OUTPUT_PATH.exists() else pd.DataFrame(columns=OUTPUT_COLS)

In [7]:
# if SUBSET == "all":
#     results = results.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

for col in OUTPUT_COLS:
    if col not in results.columns:
        results[col] = pd.NA

results["strategy"] = results["strategy"].astype("string")
results["status"] = results["status"].astype("string")
results["error"] = results["error"].astype("string")
results["seed_track"] = results["seed_track"].astype("string")
results["seed_tags"] = results["seed_tags"].astype("string")
results["rec_artist"] = results["rec_artist"].astype("string")
results["rec_album"] = results["rec_album"].astype("string")
results["rec_tags"] = results["rec_tags"].astype("string")
results["score"] = pd.to_numeric(results["score"], errors="coerce")
results["rank"] = pd.to_numeric(results["rank"], errors="coerce").astype("Int64")
results["seed_listeners"] = pd.to_numeric(results["seed_listeners"], errors="coerce").astype("Int64")
results["rec_listeners"] = pd.to_numeric(results["rec_listeners"], errors="coerce").astype("Int64")

processed_ids = set(
    results.loc[
        results["status"].notna() & (results["status"].astype(str).str.strip() != ""),
        "album_id",
    ]
)
pending = albums[~albums["album_id"].isin(processed_ids)]

print(f"Strategy: {TRACK_SELECTION}")
print(f"Pending: {len(pending):,} / {len(albums):,}")
results.head(2)

Strategy: top_listener
Pending: 19,726 / 30,516


,album_id,artist,album,review_count,strategy,status,error,seed_track,seed_listeners,rec_artist,rec_album,score,rank,rec_listeners,seed_tags,rec_tags
0,12931,Katie Got Bandz,Drillary Clinton,1,top_listener,no_results,<NA>,<NA>,2853,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>
1,10960,Indigo Girls,Poseidon and the Bitter Bug,1,top_listener,ok,<NA>,Sugar Tongue,10042,Bruce Cockburn,Guitar Gods Vol. 3 (Live),0.317808,1,8,<NA>,<NA>


In [8]:
set_request_delay(REQUEST_DELAY_SEC)

new_rows: list[dict] = []
since_save = 0

for _, row in tqdm(pending.iterrows(), total=len(pending), desc="Last.fm batch"):
    album_seed = RANDOM_SEED + int(row["album_id"])
    base = {
        "album_id": row["album_id"],
        "artist": row["artist"],
        "album": row["album"],
        "review_count": row["review_count"],
        "strategy": TRACK_SELECTION,
    }

    try:
        seed, seed_track, recs, _ = recommend_album(
            row["album"],
            artist=row["artist"],
            n_recs=N_RECS,
            fetch_floor=FETCH_FLOOR,
            track_selection=TRACK_SELECTION,
            top_n=TOP_N,
            random_seed=album_seed,
            clear_cache=False,
        )
        seed_tags = format_album_tags(seed.get("tags") or [])
        if recs.empty:
            new_rows.append({
                **base,
                "status": "no_results",
                "error": pd.NA,
                "seed_track": pd.NA,
                "seed_listeners": seed.get("listeners", pd.NA),
                "seed_tags": seed_tags,
                "rec_artist": pd.NA,
                "rec_album": pd.NA,
                "score": pd.NA,
                "rank": pd.NA,
                "rec_listeners": pd.NA,
                "rec_tags": pd.NA,
            })
        else:
            for rank, (_, rec) in enumerate(recs.iterrows(), start=1):
                if rank > N_RECS:
                    break
                try:
                    rec_listeners, rec_tag_list = get_album_listeners_and_tags(
                        rec["artist"], rec["album"]
                    )
                    rec_tags = format_album_tags(rec_tag_list)
                except Exception:
                    rec_listeners = pd.NA
                    rec_tags = pd.NA
                new_rows.append({
                    **base,
                    "status": "ok",
                    "error": pd.NA,
                    "seed_track": seed_track["name"] if seed_track else pd.NA,
                    "seed_listeners": seed.get("listeners", pd.NA),
                    "seed_tags": seed_tags,
                    "rec_artist": rec["artist"],
                    "rec_album": rec["album"],
                    "score": rec["similarity_score"],
                    "rank": rank,
                    "rec_listeners": rec_listeners,
                    "rec_tags": rec_tags,
                })
    except Exception as exc:
        new_rows.append({
            **base,
            "status": "error",
            "error": str(exc)[:500],
            "seed_track": pd.NA,
            "seed_listeners": pd.NA,
            "seed_tags": pd.NA,
            "rec_artist": pd.NA,
            "rec_album": pd.NA,
            "score": pd.NA,
            "rank": pd.NA,
            "rec_listeners": pd.NA,
            "rec_tags": pd.NA,
        })

    since_save += 1
    if since_save >= SAVE_EVERY:
        results = pd.concat([results, pd.DataFrame(new_rows)], ignore_index=True)
        new_rows = []
        results.to_csv(OUTPUT_PATH, index=False)
        since_save = 0

if new_rows:
    results = pd.concat([results, pd.DataFrame(new_rows)], ignore_index=True)
results.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")

Last.fm batch:   0%|          | 0/19726 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [8]:
print(results["status"].value_counts(dropna=False))

has_tags = (results["status"] == "ok") & results["seed_tags"].notna()
cols = ["artist", "album", "seed_track", "rank", "rec_album", "rec_artist", "score", "seed_tags", "rec_tags"]
results.loc[has_tags, cols].head(3)

status
ok            45406
error          2877
no_results     2475
Name: count, dtype: int64


,artist,album,seed_track,rank,rec_album,rec_artist,score,seed_tags,rec_tags
1,Indigo Girls,Poseidon and the Bitter Bug,Sugar Tongue,1.0,Guitar Gods Vol. 3 (Live),Bruce Cockburn,0.317808,female vocalists;singer-songwriter;folk;2009 r...,NaN
2,Indigo Girls,Poseidon and the Bitter Bug,Sugar Tongue,2.0,200 More Miles,Cowboy Junkies,0.238679,female vocalists;singer-songwriter;folk;2009 r...,alt-country;rock;female vocalists;mellow;canadian
3,Indigo Girls,Poseidon and the Bitter Bug,Sugar Tongue,3.0,Live & Alive,Tracy Chapman,0.216455,female vocalists;singer-songwriter;folk;2009 r...,folk;female vocalists;acoustic;80s;singer-song...


In [ ]:
# # Reset failed albums, then re-run the batch cell
# retry_ids = results.loc[results["status"].isin(["error", "no_results"]), "album_id"].unique()
# results = results[~results["album_id"].isin(retry_ids)]
# pending = albums[~albums["album_id"].isin(
#     results.loc[results["status"].notna(), "album_id"].unique()
# )]
# print(f"Reset {len(retry_ids):,} albums; pending: {len(pending):,} — re-run batch cell")

## Backfill Last.fm metadata

Fill missing listener/tag fields in two places (batch file first, then catalog):

1. **`lastfm_recommendations_all_top_listener.csv`** — seeds (`seed_listeners`, `seed_tags`) and recs (`rec_listeners`, `rec_tags`). **Priority:** runs first.
2. **`albums.csv`** — catalog rows still missing `lastfm_listeners` / `lastfm_tags`, ordered so batch seeds and recs come before everything else.

One `album.getinfo` call per album. Checkpoints every `SAVE_EVERY` rows. Re-run to resume.

After a large backfill, re-run `ingestion/lastfm_album_metadata.ipynb` to propagate into `final_recs.parquet`.

In [9]:
import re
import unicodedata

BACKFILL_MAX_ALBUMS = None  # e.g. 100 for a dry run (catalog phase only)
BATCH_PRIORITY_PATH = DATA_DIR / "lastfm_recommendations_all_top_listener.csv"
BACKFILL_ERRORS_PATH = DATA_DIR / "lastfm_backfill_errors.csv"

LISTENER_COL = "lastfm_listeners"
TAGS_COL = "lastfm_tags"


def normalize(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    for ch in ("\u2019", "\u2018", "\u201b", "\u2032", "`", "\u00b4"):
        s = s.replace(ch, "'")
    s = s.replace("\u2026", "...").replace("\u00b7", " ")
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s)


def _tags_missing(value) -> bool:
    return pd.isna(value)


def needs_backfill(row) -> bool:
    return pd.isna(row[LISTENER_COL]) or _tags_missing(row[TAGS_COL])


albums_catalog = pd.read_csv(SOURCE_PATH)
for col in (LISTENER_COL, TAGS_COL):
    if col not in albums_catalog.columns:
        albums_catalog[col] = pd.NA

albums_catalog[LISTENER_COL] = pd.to_numeric(
    albums_catalog[LISTENER_COL], errors="coerce"
).astype("Int64")
albums_catalog["artist_norm"] = albums_catalog["artist"].map(normalize)
albums_catalog["album_norm"] = albums_catalog["album"].map(normalize)
albums_catalog.loc[
    albums_catalog["artist_norm"].isin(["various", "va"]), "artist_norm"
] = "various artists"

batch_df = pd.read_csv(BATCH_PRIORITY_PATH)
for col in ("seed_listeners", "seed_tags", "rec_listeners", "rec_tags"):
    if col not in batch_df.columns:
        batch_df[col] = pd.NA
batch_df["seed_listeners"] = pd.to_numeric(batch_df["seed_listeners"], errors="coerce").astype("Int64")
batch_df["rec_listeners"] = pd.to_numeric(batch_df["rec_listeners"], errors="coerce").astype("Int64")
batch_df["seed_tags"] = batch_df["seed_tags"].astype("string")
batch_df["rec_tags"] = batch_df["rec_tags"].astype("string")

priority_album_ids = set(batch_df["album_id"].dropna().astype(int))
priority_rec_keys = {
    (normalize(a), normalize(b))
    for a, b in batch_df.loc[batch_df["status"] == "ok", ["rec_artist", "rec_album"]].itertuples(index=False, name=None)
    if pd.notna(a) and pd.notna(b)
}


def batch_seed_needs(row) -> bool:
    return pd.isna(row["seed_listeners"]) or _tags_missing(row["seed_tags"])


def batch_rec_needs(row) -> bool:
    if row["status"] != "ok":
        return False
    return pd.isna(row["rec_listeners"]) or _tags_missing(row["rec_tags"])


batch_seed_pending = (
    batch_df.loc[batch_df.apply(batch_seed_needs, axis=1)]
    .drop_duplicates("album_id")
    .loc[:, ["album_id", "artist", "album"]]
)
batch_rec_pending = (
    batch_df.loc[batch_df.apply(batch_rec_needs, axis=1)]
    .drop_duplicates(["rec_artist", "rec_album"])
    .loc[:, ["rec_artist", "rec_album"]]
)

backfill_pending = albums_catalog[albums_catalog.apply(needs_backfill, axis=1)].copy()


def catalog_priority(row) -> int:
    if int(row["album_id"]) in priority_album_ids:
        return 0
    if (row["artist_norm"], row["album_norm"]) in priority_rec_keys:
        return 1
    return 2


backfill_pending["_priority"] = backfill_pending.apply(catalog_priority, axis=1)
backfill_pending = backfill_pending.sort_values(["_priority", "album_id"], kind="stable")
if BACKFILL_MAX_ALBUMS is not None:
    backfill_pending = backfill_pending.head(BACKFILL_MAX_ALBUMS)

backfill_errors = (
    pd.read_csv(BACKFILL_ERRORS_PATH)
    if BACKFILL_ERRORS_PATH.exists()
    else pd.DataFrame(columns=["source", "album_id", "artist", "album", "error"])
)
if "source" not in backfill_errors.columns:
    backfill_errors.insert(0, "source", "catalog")

print(f"Batch seeds pending: {len(batch_seed_pending):,}")
print(f"Batch recs pending: {len(batch_rec_pending):,}")
print(
    f"Catalog pending: {len(backfill_pending):,} / {len(albums_catalog):,} "
    f"({(backfill_pending['_priority'] == 0).sum():,} batch seeds, "
    f"{(backfill_pending['_priority'] == 1).sum():,} batch recs in catalog, "
    f"{(backfill_pending['_priority'] == 2).sum():,} other)"
)
print(
    f"Catalog coverage: {albums_catalog[LISTENER_COL].notna().sum():,} listeners, "
    f"{albums_catalog[TAGS_COL].notna().sum():,} tags"
)
if len(backfill_errors):
    print(f"Prior errors logged: {len(backfill_errors):,} -> {BACKFILL_ERRORS_PATH}")
backfill_pending.head(2)

Batch seeds pending: 2,645
Batch recs pending: 3,423
Catalog pending: 17,087 / 30,516 (5,507 batch seeds, 2 batch recs in catalog, 11,578 other)
Catalog coverage: 13,633 listeners, 13,429 tags
Prior errors logged: 878 -> ../datasets/lastfm_backfill_errors.csv


,album_id,artist,album,review_count,genre_pitchfork,rating_mean,rating_pf,sources,n_sources,published_on_min,...,artist_area,label,catalog_number,mb_tags,mb_genre,lastfm_listeners,lastfm_tags,artist_norm,album_norm,_priority
0,0,!!!,As If,1,Rock,3.45,3.45,pitchfork,1.0,2015-10-21,...,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,!!!,as if,0
1,1,!!!,"Jamie, My Intentions Are Bass EP",1,Rock,3.40,3.40,pitchfork,1.0,2010-11-01,...,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,!!!,"jamie, my intentions are bass ep",0


In [10]:
set_request_delay(REQUEST_DELAY_SEC)

batch_new_errors: list[dict] = []
since_save = 0


def _checkpoint_batch_backfill() -> None:
    global backfill_errors
    batch_df.to_csv(BATCH_PRIORITY_PATH, index=False)
    albums_catalog.to_csv(SOURCE_PATH, index=False)
    if batch_new_errors:
        backfill_errors = pd.concat(
            [backfill_errors, pd.DataFrame(batch_new_errors)], ignore_index=True
        )
        backfill_errors.to_csv(BACKFILL_ERRORS_PATH, index=False)
        batch_new_errors.clear()


def _update_catalog_row(artist: str, album: str, listeners: int, tag_list: list[str]) -> None:
    key = (normalize(artist), normalize(album))
    match = (albums_catalog["artist_norm"] == key[0]) & (albums_catalog["album_norm"] == key[1])
    if not match.any():
        return
    idx = albums_catalog.index[match][0]
    if pd.isna(albums_catalog.at[idx, LISTENER_COL]):
        albums_catalog.at[idx, LISTENER_COL] = int(listeners)
    if _tags_missing(albums_catalog.at[idx, TAGS_COL]):
        albums_catalog.at[idx, TAGS_COL] = format_album_tags(tag_list) or ""


for _, row in tqdm(
    batch_seed_pending.iterrows(), total=len(batch_seed_pending), desc="Batch seeds"
):
    album_id = int(row["album_id"])
    mask = batch_df["album_id"] == album_id
    if not batch_df.loc[mask].apply(batch_seed_needs, axis=1).any():
        continue

    try:
        listeners, tag_list = get_album_listeners_and_tags(row["artist"], row["album"])
        tags = format_album_tags(tag_list) or ""
        needs_l = batch_df.loc[mask, "seed_listeners"].isna()
        needs_t = batch_df.loc[mask, "seed_tags"].map(_tags_missing)
        batch_df.loc[mask & needs_l, "seed_listeners"] = int(listeners)
        batch_df.loc[mask & needs_t, "seed_tags"] = tags
        cat_mask = albums_catalog["album_id"] == album_id
        if cat_mask.any():
            idx = albums_catalog.index[cat_mask][0]
            if pd.isna(albums_catalog.at[idx, LISTENER_COL]):
                albums_catalog.at[idx, LISTENER_COL] = int(listeners)
            if _tags_missing(albums_catalog.at[idx, TAGS_COL]):
                albums_catalog.at[idx, TAGS_COL] = tags
    except Exception as exc:
        batch_new_errors.append({
            "source": "batch_seed",
            "album_id": album_id,
            "artist": row["artist"],
            "album": row["album"],
            "error": str(exc)[:500],
        })

    since_save += 1
    if since_save >= SAVE_EVERY:
        _checkpoint_batch_backfill()
        since_save = 0

for _, row in tqdm(
    batch_rec_pending.iterrows(), total=len(batch_rec_pending), desc="Batch recs"
):
    mask = (
        (batch_df["status"] == "ok")
        & (batch_df["rec_artist"] == row["rec_artist"])
        & (batch_df["rec_album"] == row["rec_album"])
    )
    if not batch_df.loc[mask].apply(batch_rec_needs, axis=1).any():
        continue

    try:
        listeners, tag_list = get_album_listeners_and_tags(row["rec_artist"], row["rec_album"])
        tags = format_album_tags(tag_list) or ""
        needs_l = batch_df.loc[mask, "rec_listeners"].isna()
        needs_t = batch_df.loc[mask, "rec_tags"].map(_tags_missing)
        batch_df.loc[mask & needs_l, "rec_listeners"] = int(listeners)
        batch_df.loc[mask & needs_t, "rec_tags"] = tags
        _update_catalog_row(row["rec_artist"], row["rec_album"], listeners, tag_list)
    except Exception as exc:
        batch_new_errors.append({
            "source": "batch_rec",
            "album_id": pd.NA,
            "artist": row["rec_artist"],
            "album": row["rec_album"],
            "error": str(exc)[:500],
        })

    since_save += 1
    if since_save >= SAVE_EVERY:
        _checkpoint_batch_backfill()
        since_save = 0

_checkpoint_batch_backfill()
print(f"Saved {BATCH_PRIORITY_PATH}")
print(f"Updated catalog -> {SOURCE_PATH}")

Batch seeds:   0%|          | 0/2645 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
set_request_delay(REQUEST_DELAY_SEC)

catalog_new_errors: list[dict] = []
since_save = 0


def _checkpoint_catalog_backfill() -> None:
    global backfill_errors
    albums_catalog.to_csv(SOURCE_PATH, index=False)
    if catalog_new_errors:
        backfill_errors = pd.concat(
            [backfill_errors, pd.DataFrame(catalog_new_errors)], ignore_index=True
        )
        backfill_errors.to_csv(BACKFILL_ERRORS_PATH, index=False)
        catalog_new_errors.clear()


for idx, row in tqdm(
    backfill_pending.iterrows(), total=len(backfill_pending), desc="Catalog backfill"
):
    if not needs_backfill(albums_catalog.loc[idx]):
        continue

    try:
        listeners, tag_list = get_album_listeners_and_tags(row["artist"], row["album"])
        if pd.isna(albums_catalog.at[idx, LISTENER_COL]):
            albums_catalog.at[idx, LISTENER_COL] = int(listeners)
        if _tags_missing(albums_catalog.at[idx, TAGS_COL]):
            albums_catalog.at[idx, TAGS_COL] = format_album_tags(tag_list) or ""
    except Exception as exc:
        catalog_new_errors.append({
            "source": "catalog",
            "album_id": row["album_id"],
            "artist": row["artist"],
            "album": row["album"],
            "error": str(exc)[:500],
        })

    since_save += 1
    if since_save >= SAVE_EVERY:
        _checkpoint_catalog_backfill()
        since_save = 0

_checkpoint_catalog_backfill()

n_listeners = albums_catalog[LISTENER_COL].notna().sum()
n_tags = albums_catalog[TAGS_COL].notna().sum()
print(f"Saved {SOURCE_PATH}")
print(
    f"Coverage: {n_listeners:,} listeners ({n_listeners / len(albums_catalog):.1%}), "
    f"{n_tags:,} tags ({n_tags / len(albums_catalog):.1%})"
)
if len(backfill_errors):
    print(f"Errors logged: {len(backfill_errors):,} -> {BACKFILL_ERRORS_PATH}")

Catalog backfill:   0%|          | 0/20773 [00:00<?, ?it/s]

Saved ../datasets/albums.csv
Coverage: 30,100 listeners (98.6%), 30,100 tags (98.6%)
Errors logged: 878 -> ../datasets/lastfm_backfill_errors.csv
